<a href="https://colab.research.google.com/github/yorique/colab/blob/main/SHAPcontrato002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
TARGETS = [
    "Modelo_Desenvolvimento_2000_2019.joblib",
    "Modelo_Operacional_2000_2024.joblib",
    "Manifesto_Bloco9_V2.csv",
    "Configuracao_Bloco9_V2_Treinamento_Validacao_Caete.json",
    "Configuracao_Bloco8_V2_Tabela_Treinamento_Caete.json",
    "Tabela_Treinamento_CHIRPS_Caete_2000_2024.csv",
    "Previsoes_Teste_Final_Comparacao.csv",
    "Manifesto_Bloco21_V2.csv",
    "Inventario_Artefatos_Modelagem_Bloco21_V2.csv",
    "Calibracao_SHAP_10_Bacias_Bloco22_V2.csv",
    "Manifesto_Bloco22_V2.csv",
    "Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json",
    "Configuracao_Bloco22_V2_Interpretacao.json",
]

def sha256_readonly(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print(f"ROOT={ROOT}")
print(f"ROOT_EXISTS={ROOT.is_dir()}")

if ROOT.is_dir():
    for target in TARGETS:
        matches = sorted(path for path in ROOT.rglob(target) if path.is_file())
        print(f"TARGET={target}\tMATCHES={len(matches)}")
        for path in matches:
            print(
                f"FILE={path}\tSIZE={path.stat().st_size}\tSHA256={sha256_readonly(path)}"
            )
            if target.startswith("Configuracao_") and path.suffix == ".json":
                with path.open("r", encoding="utf-8") as handle:
                    config = json.load(handle)
                print(
                    "CONFIG="
                    + json.dumps(
                        {
                            "bloco": config.get("bloco"),
                            "versao": config.get("versao"),
                            "bacia_piloto": config.get("bacia_piloto"),
                            "bacias": config.get("bacias"),
                            "ambiente": config.get("ambiente"),
                            "modelo": config.get("modelo"),
                            "interpretacao": config.get("interpretacao"),
                        },
                        ensure_ascii=False,
                        sort_keys=True,
                    )
                )

ROOT=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2
ROOT_EXISTS=True
TARGET=Modelo_Desenvolvimento_2000_2019.joblib	MATCHES=1
FILE=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Modelos/Bloco 9 V2 - Caete/Modelo_Desenvolvimento_2000_2019.joblib	SIZE=263970426	SHA256=d22c50aa9234dcf22d309cf9f96b239d379fea2c9bf63698b335789772c6ae5a
TARGET=Modelo_Operacional_2000_2024.joblib	MATCHES=1
FILE=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Modelos/Bloco 9 V2 - Caete/Modelo_Operacional_2000_2024.joblib	SIZE=329959168	SHA256=d2314f2049b7204b34eea3fc4e04beeb465596d18161f26fa64666b33367c076
TARGET=Manifesto_Bloco9_V2.csv	MATCHES=2
FILE=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 9/Manifesto_Bloco9_V2.csv	SIZE=6996	SHA256=2ef0ec0d294a8030ebe608b33102c4bba0439f05ebbaa6c4a0acaf8a239af995
FILE=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Manifesto_Bloco9_V2.csv	SIZE=6996	SHA256=2ef0ec

In [3]:
from pathlib import Path
import csv
import importlib.metadata as metadata
import json
import platform

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
AUD = ROOT / "Auditorias"
CFG = ROOT / "Configurações"

def read_csv_rows(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        sample = handle.read(4096)
        handle.seek(0)
        dialect = csv.excel_semicolon if sample.count(";") >= sample.count(",") else csv.excel
        return list(csv.DictReader(handle, dialect=dialect))

def show_csv(path, label, focus=()):
    rows = read_csv_rows(path)
    columns = list(rows[0].keys()) if rows else []
    print(f"{label}_PATH={path}")
    print(f"{label}_ROWS={len(rows)}")
    print(f"{label}_COLUMNS={columns}")
    if focus:
        selected = [row for row in rows if any(term in str(row).lower() for term in focus)]
        print(f"{label}_FOCUS={json.dumps(selected[:20], ensure_ascii=False)}")

def first_existing(*paths):
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError(paths)

manifest_b9 = first_existing(AUD / "Manifesto_Bloco9_V2.csv", AUD / "Bloco 9" / "Manifesto_Bloco9_V2.csv")
inventory_b21 = first_existing(AUD / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv", AUD / "Bloco 21" / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv")
manifest_b21 = first_existing(AUD / "Manifesto_Bloco21_V2.csv", AUD / "Bloco 21" / "Manifesto_Bloco21_V2.csv")
calibration_b22 = first_existing(ROOT / "Tabelas Consolidadas" / "Calibracao_SHAP_10_Bacias_Bloco22_V2.csv", AUD / "Bloco 22" / "Calibracao_SHAP_10_Bacias_Bloco22_V2.csv")
manifest_b22 = first_existing(AUD / "Manifesto_Bloco22_V2.csv", AUD / "Bloco 22" / "Manifesto_Bloco22_V2.csv")

show_csv(manifest_b9, "B9_MANIFEST", ("modelo_desenvolvimento", "modelo_operacional", "previsoes_teste", "tabela_treinamento", "metadados"))
show_csv(manifest_b21, "B21_MANIFEST", ("modelo", "preprocess", "previs"))
show_csv(inventory_b21, "B21_INVENTORY", ("desenvolvimento", "preprocess", "previsoes_teste"))
show_csv(calibration_b22, "B22_CALIBRATION")
show_csv(manifest_b22, "B22_MANIFEST", ("calibracao", "importancia", "inventario", "testes"))

for name in [
    "Configuracao_Bloco9_V2_Treinamento_Validacao_Caete.json",
    "Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json",
    "Configuracao_Bloco22_V2_Interpretacao.json",
]:
    path = first_existing(CFG / name, AUD / "Bloco 9" / name, AUD / "Bloco 22" / name)
    with path.open("r", encoding="utf-8") as handle:
        cfg = json.load(handle)
    print(f"{name}_TOP_LEVEL_KEYS={sorted(cfg.keys())}")
    print(f"{name}_MODEL_KEYS={sorted((cfg.get('modelo') or {}).keys())}")

print(f"RUNTIME_PYTHON={platform.python_version()}")
for package in ["shap", "scikit-learn", "joblib"]:
    try:
        print(f"RUNTIME_{package}={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"RUNTIME_{package}=NOT_INSTALLED")

AttributeError: module 'csv' has no attribute 'excel_semicolon'

In [4]:
from pathlib import Path
import csv
import importlib.metadata as metadata
import json
import platform
from collections import Counter

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
AUD = ROOT / "Auditorias"
CFG = ROOT / "Configurações"

def first_existing(*paths):
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError(paths)

def rows(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle, delimiter=";"))

def report(path, label, terms=()):
    data = rows(path)
    print(f"{label}_PATH={path}")
    print(f"{label}_ROWS={len(data)}")
    print(f"{label}_COLUMNS={list(data[0]) if data else []}")
    selected = [row for row in data if any(term in str(row).lower() for term in terms)]
    print(f"{label}_SELECTED={json.dumps(selected[:20], ensure_ascii=False)}")
    return data

b9_manifest = report(
    first_existing(AUD / "Manifesto_Bloco9_V2.csv", AUD / "Bloco 9" / "Manifesto_Bloco9_V2.csv"),
    "B9_MANIFEST",
    ("modelo_desenvolvimento", "modelo_operacional", "previsoes_teste", "tabela_treinamento", "metadados"),
)
b21_manifest = report(
    first_existing(AUD / "Manifesto_Bloco21_V2.csv", AUD / "Bloco 21" / "Manifesto_Bloco21_V2.csv"),
    "B21_MANIFEST",
    ("modelo", "preprocess", "previs"),
)
b21_inventory = report(
    first_existing(AUD / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv", AUD / "Bloco 21" / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv"),
    "B21_INVENTORY",
    ("desenvolvimento", "preprocess", "previsoes_teste"),
)
if b21_inventory:
    print(f"B21_INVENTORY_TYPES={dict(Counter(row.get('Tipo') for row in b21_inventory))}")
    print(f"B21_INVENTORY_BACIAS={sorted(set(row.get('Bacia') for row in b21_inventory))}")

b22_calibration = report(
    first_existing(ROOT / "Tabelas Consolidadas" / "Calibracao_SHAP_10_Bacias_Bloco22_V2.csv", AUD / "Bloco 22" / "Calibracao_SHAP_10_Bacias_Bloco22_V2.csv"),
    "B22_CALIBRATION",
)
report(
    first_existing(AUD / "Manifesto_Bloco22_V2.csv", AUD / "Bloco 22" / "Manifesto_Bloco22_V2.csv"),
    "B22_MANIFEST",
    ("calibracao", "importancia", "inventario", "testes"),
)

for name in [
    "Configuracao_Bloco9_V2_Treinamento_Validacao_Caete.json",
    "Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json",
    "Configuracao_Bloco22_V2_Interpretacao.json",
]:
    path = first_existing(CFG / name, AUD / "Bloco 9" / name, AUD / "Bloco 22" / name)
    with path.open("r", encoding="utf-8") as handle:
        config = json.load(handle)
    print(f"{name}_KEYS={sorted(config)}")

print(f"RUNTIME_PYTHON={platform.python_version()}")
for package in ["shap", "scikit-learn", "joblib"]:
    try:
        print(f"RUNTIME_{package}={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"RUNTIME_{package}=NOT_INSTALLED")

B9_MANIFEST_PATH=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Manifesto_Bloco9_V2.csv
B9_MANIFEST_ROWS=27
B9_MANIFEST_COLUMNS=['Arquivo', 'Caminho', 'Existe', 'Tamanho_bytes', 'SHA256']
B9_MANIFEST_SELECTED=[{"Arquivo": "Previsoes_Teste_Final_Comparacao.csv", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 9 V2 - Modelagem Interna/Previsoes/Previsoes_Teste_Final_Comparacao.csv", "Existe": "True", "Tamanho_bytes": "1574798", "SHA256": "45866602f72535afa7b4529c889d42e5ac33ce4bc2c81bdfeae0d3e9124b1437"}, {"Arquivo": "Modelo_Desenvolvimento_2000_2019.joblib", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Modelos/Bloco 9 V2 - Caete/Modelo_Desenvolvimento_2000_2019.joblib", "Existe": "True", "Tamanho_bytes": "263970426", "SHA256": "d22c50aa9234dcf22d309cf9f96b239d379fea2c9bf63698b335789772c6ae5a"}, {"Arquivo": "Modelo_Operacional_2000_2024.joblib", "Caminho": "/content/drive/MyDrive/Meus 

In [5]:
from pathlib import Path
import csv
import json
from collections import Counter

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
AUD = ROOT / "Auditorias"
CFG = ROOT / "Configurações"

def first_existing(*paths):
    return next(path for path in paths if path.is_file())

def load_rows(path):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle, delimiter=";"))

def check_paths(rows):
    checked = []
    for row in rows:
        path = Path(row["Caminho"])
        exists = path.is_file()
        size_ok = exists and str(path.stat().st_size) == str(row.get("Tamanho_bytes", "")).strip()
        checked.append((exists, size_ok))
    return {
        "total": len(checked),
        "exists": sum(exists for exists, _ in checked),
        "size_match": sum(size_ok for _, size_ok in checked),
    }

b9_manifest = load_rows(first_existing(AUD / "Manifesto_Bloco9_V2.csv", AUD / "Bloco 9" / "Manifesto_Bloco9_V2.csv"))
b21_inventory = load_rows(first_existing(AUD / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv", AUD / "Bloco 21" / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv"))
b22_manifest = load_rows(first_existing(AUD / "Manifesto_Bloco22_V2.csv", AUD / "Bloco 22" / "Manifesto_Bloco22_V2.csv"))
print("B9_PATHS=", check_paths(b9_manifest))
print("B21_PATHS=", check_paths(b21_inventory))
print("B22_PATHS=", check_paths(b22_manifest))

with (CFG / "Configuracao_Bloco9_V2_Treinamento_Validacao_Caete.json").open(encoding="utf-8") as handle:
    b9_config = json.load(handle)
print("B9_MODEL_SELECTION=", json.dumps(b9_config.get("modelo_selecionado"), ensure_ascii=False))
print("B9_MODEL_ARTIFACTS=", json.dumps(b9_config.get("artefatos_modelo"), ensure_ascii=False))
print("B9_DATA=", json.dumps(b9_config.get("dados"), ensure_ascii=False))
print("B9_MISSINGNESS=", json.dumps(b9_config.get("ausencias"), ensure_ascii=False))

metadata_path = ROOT / "Modelos" / "Bloco 9 V2 - Caete" / "Metadados_Modelo_Selecionado.json"
with metadata_path.open(encoding="utf-8") as handle:
    selected_metadata = json.load(handle)
print("B9_SELECTED_METADATA=", json.dumps(selected_metadata, ensure_ascii=False))

calibration_path = ROOT / "Tabelas Consolidadas" / "Calibracao_SHAP_10_Bacias_Bloco22_V2.csv"
calibration = load_rows(calibration_path)
print("B22_CALIBRATION_ROWS=", json.dumps(calibration, ensure_ascii=False))

for test_name, paths in {
    "B9_TESTS": [AUD / "Testes_Bloco9_V2.csv", AUD / "Bloco 9" / "Testes_Bloco9_V2.csv"],
    "B22_TESTS": [AUD / "Testes_Bloco22_V2_Interpretacao.csv", AUD / "Bloco 22" / "Testes_Bloco22_V2_Interpretacao.csv"],
}.items():
    test_rows = load_rows(first_existing(*paths))
    approved = sum(str(row.get("Aprovado", "")).strip().lower() in {"true", "sim", "1"} for row in test_rows)
    print(f"{test_name}_SUMMARY={{'total': {len(test_rows)}, 'approved': {approved}}}")

B9_PATHS= {'total': 27, 'exists': 27, 'size_match': 27}
B21_PATHS= {'total': 130, 'exists': 130, 'size_match': 130}
B22_PATHS= {'total': 12, 'exists': 12, 'size_match': 12}
B9_MODEL_SELECTION= {"candidato": {"id": "ET_FULL_TENDENCIA_RAW", "escopo": "FULL", "algoritmo": "EXTRA_TREES", "conjunto": "FULL_TENDENCIA", "transformacao": "RAW", "parametros": {"n_estimators": 400, "max_features": 0.65, "min_samples_leaf": 2, "min_samples_split": 2, "max_depth": null}}, "variaveis": ["TCC", "TCWV", "RH2M", "Q2M", "SP", "MSLP", "SLHF", "CAPE", "MFD", "U10", "V10", "WS10", "WIND_DOWNSLOPE_MPS", "WIND_UPSLOPE_MPS", "WIND_CROSS_MPS", "WIND_ALIGNMENT_COS", "ORO_LIFT_PROXY", "ORO_MOISTURE_PROXY", "NDVI", "EVI", "LST_DAY", "LST_NIGHT", "LST_AMP", "ET", "LE", "PET", "PLE", "LAI", "FPAR", "ALB_BSA", "ALB_WSA", "ELEVATION_M", "SLOPE_DEG", "SLOPE_PCT", "ASPECT_DEG", "ASPECT_SIN", "ASPECT_COS", "TPI_1KM_M", "TPI_5KM_M", "RELIEF_1KM_M", "RELIEF_5KM_M", "HAND_M", "UPA_KM2", "LOG_UPA", "TWI_PROXY", "DIST_DRAIN

In [6]:
from pathlib import Path
import csv
import gzip

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")

def header(path):
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8-sig", newline="") as handle:
        return next(csv.reader(handle, delimiter=";"))

files = {
    "B9_TRAINING": ROOT / "Tabelas Consolidadas" / "Treinamento Caete" / "Tabela_Treinamento_CHIRPS_Caete_2000_2024.csv",
    "B9_PREDICTIONS": ROOT / "Auditorias" / "Bloco 9 V2 - Modelagem Interna" / "Previsoes" / "Previsoes_Teste_Final_Comparacao.csv",
    "B10_SAMPLE": ROOT / "Auditorias" / "Bloco 10 V2 - SHAP Permutacao" / "Amostra_SHAP_Teste_Final.csv",
    "B10_CALIBRATION": ROOT / "Auditorias" / "Bloco 10 V2 - SHAP Permutacao" / "Calibracao_SHAP_Aproximado_vs_Exato.csv",
    "B10_LOCAL_VALUES": ROOT / "Auditorias" / "Bloco 10 V2 - SHAP Permutacao" / "Valores_SHAP_Teste_Final_Amostra.parquet",
    "B21_APEU_TEST": ROOT / "Modelos" / "Fase B - Modelos Independentes" / "apeu" / "Teste_Selecionado_apeu_B21.csv.gz",
    "B21_APEU_PREDICTIONS": ROOT / "Modelos" / "Fase B - Modelos Independentes" / "apeu" / "Previsoes_Teste_Selecionado_apeu_B21.csv.gz",
}
for label, path in files.items():
    print(f"{label}_EXISTS={path.is_file()}")
    if path.is_file() and path.suffix in {".csv", ".gz"}:
        print(f"{label}_HEADER={header(path)}")

B9_TRAINING_EXISTS=True
B9_TRAINING_HEADER=['row_id', 'Bacia', 'cell_id', 'chirps_row', 'chirps_col', 'centro_lon', 'centro_lat', 'centro_na_bacia', 'area_celula_km2', 'area_na_bacia_km2', 'fracao_bacia', 'Peso_Area_Bacia', 'selecionada_treino', 'Fold_Espacial', 'Grupo_Espacial', 'Ano', 'Mes', 'Ano_Mes', 'Data', 'Papel_Temporal', 'Teste_Final', 'Fold_Temporal_Ano', 'Indice_Tempo', 'Indice_Tempo_Norm', 'Mes_Sin', 'Mes_Cos', 'PRECIP_MM', 'TCC', 'TCWV', 'RH2M', 'Q2M', 'SP', 'MSLP', 'SLHF', 'CAPE', 'MFD', 'U10', 'V10', 'WS10', 'WIND_DOWNSLOPE_MPS', 'WIND_UPSLOPE_MPS', 'WIND_CROSS_MPS', 'WIND_ALIGNMENT_COS', 'ORO_LIFT_PROXY', 'ORO_MOISTURE_PROXY', 'NDVI', 'EVI', 'LST_DAY', 'LST_NIGHT', 'LST_AMP', 'ET', 'LE', 'PET', 'PLE', 'LAI', 'FPAR', 'ALB_BSA', 'ALB_WSA', 'COV_NDVI', 'COV_EVI', 'COV_LST_DAY', 'COV_LST_NIGHT', 'COV_LST_AMP', 'COV_ET', 'COV_LE', 'COV_PET', 'COV_PLE', 'COV_LAI', 'COV_FPAR', 'COV_ALB_BSA', 'COV_ALB_WSA', 'ELEVATION_M', 'SLOPE_DEG', 'SLOPE_PCT', 'ASPECT_DEG', 'ASPECT_SIN', 'A

In [7]:
from pathlib import Path
import csv

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
NAMES = [
    "Amostra_SHAP_Teste_Final.csv",
    "Calibracao_SHAP_Aproximado_vs_Exato.csv",
    "Valores_SHAP_Teste_Final_Amostra.parquet",
    "Checkpoint_SHAP_Teste_Final.joblib",
    "Teste_Selecionado_apeu_B21.csv.gz",
]
for name in NAMES:
    matches = sorted(path for path in ROOT.rglob(name) if path.is_file())
    print(f"SEARCH={name}\tMATCHES={len(matches)}")
    for path in matches:
        print(f"FOUND={path}\tSIZE={path.stat().st_size}")

inventory = ROOT / "Auditorias" / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv"
with inventory.open("r", encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle, delimiter=";"))
for artifact_type in ["teste", "variaveis", "metadados", "desenvolvimento", "previsoes_teste"]:
    selected = [row for row in rows if row.get("Tipo") == artifact_type]
    print(f"B21_TYPE={artifact_type}\tCOUNT={len(selected)}")
    for row in selected:
        path = Path(row["Caminho"])
        print(f"B21_FILE={row['Bacia']}\t{path.name}\tEXISTS={path.is_file()}\tSIZE={path.stat().st_size if path.is_file() else None}")

SEARCH=Amostra_SHAP_Teste_Final.csv	MATCHES=1
FOUND=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Tabelas/Amostra_SHAP_Teste_Final.csv	SIZE=190883
SEARCH=Calibracao_SHAP_Aproximado_vs_Exato.csv	MATCHES=1
FOUND=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Tabelas/Calibracao_SHAP_Aproximado_vs_Exato.csv	SIZE=4526
SEARCH=Valores_SHAP_Teste_Final_Amostra.parquet	MATCHES=2
FOUND=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10/Valores_SHAP_Teste_Final_Amostra.parquet	SIZE=963504
FOUND=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Tabelas/Valores_SHAP_Teste_Final_Amostra.parquet	SIZE=963504
SEARCH=Checkpoint_SHAP_Teste_Final.joblib	MATCHES=1
FOUND=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Checkpoints/Checkpoint_SHAP_Test

In [8]:
from pathlib import Path
import csv
from collections import Counter

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
candidates = list(ROOT.rglob("Inventario_Tabelas_Treinamento_Operacionais_Bloco20_V2.csv"))
print(f"B20_INVENTORY_MATCHES={len(candidates)}")
for path in candidates:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle, delimiter=";"))
    exists = sum(Path(row["Caminho"]).is_file() for row in rows)
    size_match = sum(Path(row["Caminho"]).is_file() and str(Path(row["Caminho"]).stat().st_size) == str(row["Tamanho_bytes"]).strip() for row in rows)
    print(f"B20_INVENTORY={path}")
    print(f"B20_ROWS={len(rows)} EXISTS={exists} SIZE_MATCH={size_match}")
    print(f"B20_TYPES={dict(Counter(row.get('Tipo') for row in rows))}")
    print(f"B20_BACIAS={sorted(set(row.get('Bacia') for row in rows))}")

B20_INVENTORY_MATCHES=2
B20_INVENTORY=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Inventario_Tabelas_Treinamento_Operacionais_Bloco20_V2.csv
B20_ROWS=20 EXISTS=20 SIZE_MATCH=20
B20_TYPES={'TREINAMENTO': 10, 'MOD17_SENSIBILIDADE': 10}
B20_BACIAS=['apeu', 'caripi', 'guama', 'maracana', 'marapanim', 'morcego', 'piria', 'quatipuru', 'taperacu', 'urumajo']
B20_INVENTORY=/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 20/Inventario_Tabelas_Treinamento_Operacionais_Bloco20_V2.csv
B20_ROWS=20 EXISTS=20 SIZE_MATCH=20
B20_TYPES={'TREINAMENTO': 10, 'MOD17_SENSIBILIDADE': 10}
B20_BACIAS=['apeu', 'caripi', 'guama', 'maracana', 'marapanim', 'morcego', 'piria', 'quatipuru', 'taperacu', 'urumajo']


In [9]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
matches = sorted(
    path for path in ROOT.rglob("*")
    if path.is_file()
    and any(token in path.name.lower() for token in ("checkpoint", "calibracao_shap", "amostra_shap"))
)
print(f"SHAP_RELATED_FILES={len(matches)}")
for path in matches:
    print(f"{path}\t{path.stat().st_size}")

SHAP_RELATED_FILES=69
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Interpretacao dos Modelos/apeu/Calibracao_SHAP_apeu_B22.csv	315
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Interpretacao dos Modelos/caripi/Calibracao_SHAP_caripi_B22.csv	317
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Interpretacao dos Modelos/guama/Calibracao_SHAP_guama_B22.csv	315
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Interpretacao dos Modelos/maracana/Calibracao_SHAP_maracana_B22.csv	319
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Interpretacao dos Modelos/marapanim/Calibracao_SHAP_marapanim_B22.csv	320
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Interpretacao dos Modelos/morcego/Calibracao_SHAP_morcego_

In [10]:
from pathlib import Path
import csv
import json

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
checkpoint_dir = ROOT / "Staging" / "Bloco_22_V2_Interpretacao"
calibration_dir = ROOT / "Artefatos Operacionais" / "Fase B - Interpretacao dos Modelos"

for checkpoint in sorted(checkpoint_dir.glob("Checkpoint_B22_*.json")):
    with checkpoint.open(encoding="utf-8") as handle:
        data = json.load(handle)
    print(f"CHECKPOINT={checkpoint.name}\tKEYS={sorted(data)}\tDATA={json.dumps(data, ensure_ascii=False)}")

for path in sorted(calibration_dir.glob("*/Calibracao_SHAP_*_B22.csv")):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        data = list(csv.DictReader(handle, delimiter=";"))
    print(f"CALIBRATION={path.name}\tROWS={len(data)}\tCOLUMNS={list(data[0]) if data else []}\tDATA={json.dumps(data, ensure_ascii=False)}")

CHECKPOINT=Checkpoint_B22_apeu.json	KEYS=['artefatos', 'identidade']	DATA={"identidade": {"versao": "BLOCO_22_V2_v1.0.0", "bacia": "apeu", "sha_tabela_b20": "a874f39bd2b4a93d01aa3677f158235d8d0b00bd909ad45b08f9cd50fe5b6ea3", "sha_modelo_dev": "e0f96727f843b2cdc56d6462275c972b127ed078e4778d7fb14f77e93d805b62", "shap_por_ano_mes_fold": 4, "perm_repeticoes": 3}, "artefatos": {"importancia": {"tamanho_bytes": 7595, "sha256": "f1680b2bb280b3c790856f5fb4693a792a3b91478d169821231c0a0fd4d485a1"}, "grupos": {"tamanho_bytes": 613, "sha256": "01bf059f99d6a701a83b34ab960cbb766ef13170abbad28f9e42d27037940b2e"}, "correlacoes": {"tamanho_bytes": 856, "sha256": "121205648641eca194ed2c5e23330aa8684c22699cdbbfe956a1b311960fc0b1"}, "envelopes": {"tamanho_bytes": 7530, "sha256": "1b9a1eb7de696fcd248adabf2099df26323c10624a1c806e60f80dff0a144597"}, "calibracao": {"tamanho_bytes": 315, "sha256": "325406e183a8f1f754232adf59d52f7f1a81c0b0e0dec0f4252d270c985e946a"}, "metadados": {"tamanho_bytes": 1334, "sha256"

In [11]:
from pathlib import Path
import re

PATTERNS = ("Calibracao_SHAP", "SEMENTE_ALEATORIA", "shap_por_ano_mes_fold", "amostra_estratificada")
hits = []
for path in ROOT.rglob("*"):
    if path.suffix.lower() not in {".py", ".ipynb", ".md", ".txt"}:
        continue
    if path.stat().st_size > 20_000_000:
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    found = [p for p in PATTERNS if p.lower() in text.lower()]
    if found:
        hits.append((str(path), path.stat().st_size, found))
print("SOURCE_HITS", len(hits))
for item in hits[:100]:
    print("HIT", item)


SOURCE_HITS 2
HIT ('/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Relatórios/Relatorio_Bloco22_V2_Interpretacao.txt', 5735, ['Calibracao_SHAP'])
HIT ('/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 22/Relatorio_Bloco22_V2_Interpretacao.txt', 5735, ['Calibracao_SHAP'])


In [12]:
report_b22 = ROOT / "Relatórios" / "Relatorio_Bloco22_V2_Interpretacao.txt"
print(report_b22.read_text(encoding="utf-8", errors="replace"))

RELATORIO - BLOCO 22 V2
INTERPRETACAO DOS MODELOS E ENVELOPES DE EXTRAPOLACAO

Data UTC: 2026-08-18T19:33:05+00:00
Bacias interpretadas: 10.
Selecao B21 reaberta: NAO.

AUDITORIA B21
----------------------------------------------------------------------------------------------------------------------
Testes B21: 20/20.
Artefatos compactos B21: 11/11.
Artefatos de modelagem B21: 130/130 revalidados.

INTERPRETACAO
----------------------------------------------------------------------------------------------------------------------
Modelo: desenvolvimento 2000-2019.
Periodo principal: teste final 2020-2024.
SHAP: exato + calibracao aproximada + fallback exato.
Permutacao: GLOBAL e DENTRO_ANO_MES.
Repeticoes de permutacao: 3.
Correlacao: Spearman no desenvolvimento.
MDI: diagnostico complementar Extra Trees.

RESUMO POR BACIA
----------------------------------------------------------------------------------------------------------------------
    Bacia             Candidato  N_Variaveis  

In [13]:
for path in sorted(ROOT.rglob("*Inventario*.csv")):
    if "20" in path.name or "21" in path.name or "B20" in str(path) or "B21" in str(path):
        print(path)

/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 1/Inventario_Subclasses_Caete_2000_2024_Bloco1_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 20/Inventario_Conjuntos_Modelagem_Bloco20_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 20/Inventario_Janelas_Temporais_Bloco20_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 20/Inventario_Tabelas_Treinamento_Operacionais_Bloco20_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 21/Inventario_Artefatos_Modelagem_Bloco21_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Inventario_Artefatos_Modelagem_Bloco21_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Inventario_Conjuntos_Modelagem_Bloco20_V2.csv
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Inventario_Janela

In [14]:
import csv, json
p20 = ROOT / "Auditorias" / "Bloco 20" / "Inventario_Conjuntos_Modelagem_Bloco20_V2.csv"
p21 = ROOT / "Auditorias" / "Bloco 21" / "Inventario_Artefatos_Modelagem_Bloco21_V2.csv"
for tag, path in [("B20", p20), ("B21", p21)]:
    with path.open(encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f, delimiter=";"))
    print(tag, "HEADER", list(rows[0]))
    print(tag, "FIRST3", json.dumps(rows[:3], ensure_ascii=False))
print("B9_CANDIDATES")
for p in ROOT.rglob("*"):
    s = str(p)
    if p.is_file() and ("Bloco 9" in s or "Bloco9" in s) and p.suffix.lower() in {".csv", ".json", ".joblib"}:
        print(p, p.stat().st_size)

B20 HEADER ['Conjunto', 'Ordem', 'Variavel', 'Papel']
B20 FIRST3 [{"Conjunto": "FULL_FISICO", "Ordem": "1", "Variavel": "TCC", "Papel": "PREDITOR"}, {"Conjunto": "FULL_FISICO", "Ordem": "2", "Variavel": "TCWV", "Papel": "PREDITOR"}, {"Conjunto": "FULL_FISICO", "Ordem": "3", "Variavel": "RH2M", "Papel": "PREDITOR"}]
B21 HEADER ['Bacia', 'Tipo', 'Arquivo', 'Caminho', 'Tamanho_bytes', 'SHA256']
B21 FIRST3 [{"Bacia": "apeu", "Tipo": "baselines", "Arquivo": "Baselines_Referencia_Teste_apeu_B21.csv", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Modelos/Fase B - Modelos Independentes/apeu/Baselines_Referencia_Teste_apeu_B21.csv", "Tamanho_bytes": "1814", "SHA256": "2b74adb2fbae82220f6f0bbab08e412f12aa3d595602a745f3ca1a0a06be66bb"}, {"Bacia": "apeu", "Tipo": "desenvolvimento", "Arquivo": "Modelo_Desenvolvimento_apeu_B21.joblib", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Modelos/Fase B - Modelos Independentes/apeu/Modelo_Desenvolv

In [15]:
p20files = ROOT / "Auditorias" / "Bloco 20" / "Inventario_Tabelas_Treinamento_Operacionais_Bloco20_V2.csv"
with p20files.open(encoding="utf-8-sig", newline="") as f:
    rows20files = list(csv.DictReader(f, delimiter=";"))
print("B20_TABLES_HEADER", list(rows20files[0]))
print("B20_TABLES_DATA", json.dumps(rows20files, ensure_ascii=False))

B20_TABLES_HEADER ['Bacia', 'Tipo', 'Arquivo', 'Caminho', 'Linhas', 'Tamanho_bytes', 'SHA256']
B20_TABLES_DATA [{"Bacia": "apeu", "Tipo": "TREINAMENTO", "Arquivo": "Tabela_Treinamento_CHIRPS_apeu_2000_2024.csv.gz", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Tabelas de Treinamento/apeu/Tabela_Treinamento_CHIRPS_apeu_2000_2024.csv.gz", "Linhas": "14400", "Tamanho_bytes": "6437305", "SHA256": "a874f39bd2b4a93d01aa3677f158235d8d0b00bd909ad45b08f9cd50fe5b6ea3"}, {"Bacia": "apeu", "Tipo": "MOD17_SENSIBILIDADE", "Arquivo": "Tabela_Sensibilidade_MOD17_apeu_2021_2024.csv.gz", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/Fase B - Tabelas de Treinamento/apeu/Tabela_Sensibilidade_MOD17_apeu_2021_2024.csv.gz", "Linhas": "2304", "Tamanho_bytes": "1074794", "SHA256": "a6514fad2bbf07c941db9541afcf9b6d498807c702d67b138949d085a4b1b932"}, {"Bacia": "caripi", "Tipo": "TREINAMENTO", "Arqui

In [16]:
import hashlib, sys

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

with p21.open(encoding="utf-8-sig", newline="") as f:
    rows21 = list(csv.DictReader(f, delimiter=";"))
preflight_hashes = {}
for table_row in rows20files:
    if table_row["Tipo"] != "TREINAMENTO":
        continue
    basin = table_row["Bacia"]
    relevant21 = {r["Tipo"]: r for r in rows21 if r["Bacia"] == basin and r["Tipo"] in {"desenvolvimento", "previsoes_teste"}}
    files = [
        ("tabela_b20", table_row["Caminho"], table_row["SHA256"]),
        ("modelo_b21", relevant21["desenvolvimento"]["Caminho"], relevant21["desenvolvimento"]["SHA256"]),
        ("previsoes_b21", relevant21["previsoes_teste"]["Caminho"], relevant21["previsoes_teste"]["SHA256"]),
    ]
    results = {}
    for label, raw_path, expected in files:
        actual = sha256_file(Path(raw_path))
        results[label] = {"expected": expected, "actual": actual, "ok": actual == expected}
    cp = json.loads((checkpoint_dir / f"Checkpoint_B22_{basin}.json").read_text(encoding="utf-8"))
    results["checkpoint_table_match"] = cp["identidade"]["sha_tabela_b20"] == table_row["SHA256"]
    results["checkpoint_model_match"] = cp["identidade"]["sha_modelo_dev"] == relevant21["desenvolvimento"]["SHA256"]
    preflight_hashes[basin] = results
    print(basin, "OK" if all(v["ok"] for v in results.values() if isinstance(v, dict)) and results["checkpoint_table_match"] and results["checkpoint_model_match"] else "FAIL", json.dumps(results, ensure_ascii=False))
    sys.stdout.flush()
print("HASH_PREFLIGHT_COMPLETE", len(preflight_hashes), "basins")

apeu OK {"tabela_b20": {"expected": "a874f39bd2b4a93d01aa3677f158235d8d0b00bd909ad45b08f9cd50fe5b6ea3", "actual": "a874f39bd2b4a93d01aa3677f158235d8d0b00bd909ad45b08f9cd50fe5b6ea3", "ok": true}, "modelo_b21": {"expected": "e0f96727f843b2cdc56d6462275c972b127ed078e4778d7fb14f77e93d805b62", "actual": "e0f96727f843b2cdc56d6462275c972b127ed078e4778d7fb14f77e93d805b62", "ok": true}, "previsoes_b21": {"expected": "73cbb58f9abfce3d3d78c83c84aa568e70ec978be2dfde1cbb3954be3450b324", "actual": "73cbb58f9abfce3d3d78c83c84aa568e70ec978be2dfde1cbb3954be3450b324", "ok": true}, "checkpoint_table_match": true, "checkpoint_model_match": true}
caripi OK {"tabela_b20": {"expected": "c080903841f3f13db48c661695ab733ce7ee358b1dc7f7f116cc39ebbb18a21c", "actual": "c080903841f3f13db48c661695ab733ce7ee358b1dc7f7f116cc39ebbb18a21c", "ok": true}, "modelo_b21": {"expected": "11a072e31c46b54a8eab98377c36def452d020eb8740231fc60d56f862a3229d", "actual": "11a072e31c46b54a8eab98377c36def452d020eb8740231fc60d56f862a3229

In [17]:
p9manifest = ROOT / "Auditorias" / "Manifesto_Bloco9_V2.csv"
with p9manifest.open(encoding="utf-8-sig", newline="") as f:
    rows9manifest = list(csv.DictReader(f, delimiter=";"))
print("B9_MANIFEST_HEADER", list(rows9manifest[0]))
for row in rows9manifest:
    label = " ".join(str(v) for v in row.values()).lower()
    if any(token in label for token in ("modelo_desenvolvimento", "tabela_treinamento", "previsoes_teste_final")):
        print(json.dumps(row, ensure_ascii=False))

B9_MANIFEST_HEADER ['Arquivo', 'Caminho', 'Existe', 'Tamanho_bytes', 'SHA256']
{"Arquivo": "Previsoes_Teste_Final_Comparacao.csv", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 9 V2 - Modelagem Interna/Previsoes/Previsoes_Teste_Final_Comparacao.csv", "Existe": "True", "Tamanho_bytes": "1574798", "SHA256": "45866602f72535afa7b4529c889d42e5ac33ce4bc2c81bdfeae0d3e9124b1437"}
{"Arquivo": "Modelo_Desenvolvimento_2000_2019.joblib", "Caminho": "/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Modelos/Bloco 9 V2 - Caete/Modelo_Desenvolvimento_2000_2019.joblib", "Existe": "True", "Tamanho_bytes": "263970426", "SHA256": "d22c50aa9234dcf22d309cf9f96b239d379fea2c9bf63698b335789772c6ae5a"}


In [18]:
b9_targets = [r for r in rows9manifest if r["Arquivo"] in {"Modelo_Desenvolvimento_2000_2019.joblib", "Previsoes_Teste_Final_Comparacao.csv"}]
b9_hash_results = {}
for r in b9_targets:
    actual = sha256_file(Path(r["Caminho"]))
    b9_hash_results[r["Arquivo"]] = {"expected": r["SHA256"], "actual": actual, "ok": actual == r["SHA256"]}
print("caete", "OK" if all(v["ok"] for v in b9_hash_results.values()) else "FAIL", json.dumps(b9_hash_results, ensure_ascii=False))

caete OK {"Previsoes_Teste_Final_Comparacao.csv": {"expected": "45866602f72535afa7b4529c889d42e5ac33ce4bc2c81bdfeae0d3e9124b1437", "actual": "45866602f72535afa7b4529c889d42e5ac33ce4bc2c81bdfeae0d3e9124b1437", "ok": true}, "Modelo_Desenvolvimento_2000_2019.joblib": {"expected": "d22c50aa9234dcf22d309cf9f96b239d379fea2c9bf63698b335789772c6ae5a", "actual": "d22c50aa9234dcf22d309cf9f96b239d379fea2c9bf63698b335789772c6ae5a", "ok": true}}


In [19]:
import gc
import joblib
import sklearn
import shap
import warnings

def describe_artifact(label, path):
    result = {"label": label, "load_ok": False}
    try:
        artifact = joblib.load(path)
        result["load_ok"] = True
        result["artifact_type"] = type(artifact).__name__
        if isinstance(artifact, dict):
            result["keys"] = sorted(map(str, artifact.keys()))
            pipe = artifact.get("pipeline") or artifact.get("modelo") or artifact.get("model")
        else:
            pipe = artifact
        result["pipeline_type"] = type(pipe).__name__ if pipe is not None else None
        if hasattr(pipe, "named_steps"):
            result["steps"] = {name: type(step).__name__ for name, step in pipe.named_steps.items()}
            model_steps = [name for name, step in pipe.named_steps.items() if hasattr(step, "predict")]
            prep_steps = [name for name, step in pipe.named_steps.items() if hasattr(step, "transform") and not hasattr(step, "predict")]
            result["model_steps"] = model_steps
            result["preprocessor_steps"] = prep_steps
        del artifact
        gc.collect()
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    return result

model_paths = {"caete": b9_targets[[r["Arquivo"] for r in b9_targets].index("Modelo_Desenvolvimento_2000_2019.joblib")]["Caminho"]}
model_paths.update({r["Bacia"]: r["Caminho"] for r in rows21 if r["Tipo"] == "desenvolvimento"})
model_preproc_preflight = {}
print("RUNTIME", {"python": sys.version.split()[0], "sklearn": sklearn.__version__, "joblib": joblib.__version__, "shap": shap.__version__})
for basin, path in model_paths.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = describe_artifact(basin, path)
    result["warnings"] = [f"{type(w.message).__name__}: {w.message}" for w in caught[:3]]
    model_preproc_preflight[basin] = result
    print(basin, json.dumps(result, ensure_ascii=False))
    sys.stdout.flush()
print("MODEL_PREPROCESSOR_PREFLIGHT_COMPLETE", len(model_preproc_preflight))

RUNTIME {'python': '3.13.18', 'sklearn': '1.6.1', 'joblib': '1.6.0', 'shap': '0.54.1'}
caete {"label": "caete", "load_ok": true, "artifact_type": "dict", "keys": ["alvo", "candidato", "clip_minimo", "hash_tabela_entrada", "metricas_teste", "periodo_teste", "periodo_treino", "pipeline", "selecao_usou_teste_final", "variaveis", "versao"], "pipeline_type": "Pipeline", "steps": {"preprocessador": "Pipeline", "modelo": "ExtraTreesRegressor"}, "model_steps": ["modelo"], "preprocessor_steps": ["preprocessador"], "warnings": ["DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC)."]}
apeu {"label": "apeu", "load_ok": true, "artifact_type": "dict", "keys": ["alvo", "assinatura_grade_candidatos", "bacia", "candidato", "clip_minimo", "hash_tabela_entrada", "metricas_teste", "periodo_teste", "periodo_treino", "pipeline", "selecao_usou_teste_final", "va

In [20]:
def detailed_model_check(basin, path, expected_table_hash=None):
    result = {"basin": basin}
    artifact = joblib.load(path)
    pipe = artifact["pipeline"]
    prep = pipe.named_steps["preprocessador"]
    result["candidate"] = artifact.get("candidato")
    result["n_variables"] = len(artifact.get("variaveis", []))
    result["input_table_hash"] = artifact.get("hash_tabela_entrada")
    result["table_hash_matches"] = (artifact.get("hash_tabela_entrada") == expected_table_hash) if expected_table_hash else None
    result["preprocessor_type"] = type(prep).__name__
    result["preprocessor_steps"] = {n: type(s).__name__ for n, s in prep.named_steps.items()} if hasattr(prep, "named_steps") else {}
    result["n_features_in"] = int(getattr(prep, "n_features_in_", -1))
    result["variable_count_matches_preprocessor"] = result["n_variables"] == result["n_features_in"]
    del artifact
    gc.collect()
    return result

expected_b20_hash = {r["Bacia"]: r["SHA256"] for r in rows20files if r["Tipo"] == "TREINAMENTO"}
detailed_preflight = {}
for basin, path in model_paths.items():
    result = detailed_model_check(basin, path, expected_b20_hash.get(basin))
    detailed_preflight[basin] = result
    print(json.dumps(result, ensure_ascii=False))
    sys.stdout.flush()
print("DETAILED_MODEL_PREFLIGHT_COMPLETE", len(detailed_preflight))

{"basin": "caete", "candidate": {"id": "ET_FULL_TENDENCIA_RAW", "escopo": "FULL", "algoritmo": "EXTRA_TREES", "conjunto": "FULL_TENDENCIA", "transformacao": "RAW", "parametros": {"n_estimators": 400, "max_features": 0.65, "min_samples_leaf": 2, "min_samples_split": 2, "max_depth": null}}, "n_variables": 53, "input_table_hash": "b8fb61df23f39cab2587af6605b5a429cd1125e40641acf5758df4c083a4a55c", "table_hash_matches": null, "preprocessor_type": "Pipeline", "preprocessor_steps": {"imputador": "SimpleImputer", "variancia": "VarianceThreshold"}, "n_features_in": 53, "variable_count_matches_preprocessor": true}
{"basin": "apeu", "candidate": {"id": "ET_FULL_TENDENCIA_RAW", "escopo": "FULL", "algoritmo": "EXTRA_TREES", "conjunto": "FULL_TENDENCIA", "transformacao": "RAW", "parametros": {"n_estimators": 400, "max_features": 0.65, "min_samples_leaf": 2, "min_samples_split": 2, "max_depth": null}}, "n_variables": 53, "input_table_hash": "a874f39bd2b4a93d01aa3677f158235d8d0b00bd909ad45b08f9cd50fe5

In [21]:
from pathlib import Path
import hashlib, json, re, sys
import pandas as pd
import requests

print("ISOLATED_NOTEBOOK=scratchpad")
print("DRIVE_ROOT_ACCESSIBLE=", ROOT.is_dir())
print("PANDAS_RUNTIME=", pd.__version__)

for term in ("Parecer", "ATE", "Bloco 10", "Bloco 22"):
    matches = [p for p in ROOT.rglob("*") if p.is_file() and term.lower() in p.name.lower()]
    print(f"LOCAL_SEARCH_{term}={len(matches)}")
    for p in matches[:30]:
        print(p)

source_ids = {
    "B10": "1VdWc3flW7KcPpt79hQjwiIJZMUQIzUt1",
    "B22": "1Pt-LtiLa_RtvIRha6jZ93VTvAJJzobdg",
}
historical_source_text = {}
for name, file_id in source_ids.items():
    response = requests.get(
        "https://drive.usercontent.google.com/download",
        params={"id": file_id, "export": "download", "confirm": "t"},
        timeout=60,
    )
    payload = response.content
    print(name, "HTTP", response.status_code, "CONTENT_TYPE", response.headers.get("content-type"), "BYTES", len(payload), "SHA256", hashlib.sha256(payload).hexdigest())
    text = payload.decode("utf-8", errors="replace")
    historical_source_text[name] = text
    print(name, "HAS_SEMENTE", "SEMENTE_ALEATORIA" in text, "HAS_AMOSTRA", "amostra_estratificada" in text, "PREFIX", repr(text[:80]))

ISOLATED_NOTEBOOK=scratchpad
DRIVE_ROOT_ACCESSIBLE= True
PANDAS_RUNTIME= 2.2.3
LOCAL_SEARCH_Parecer=0
LOCAL_SEARCH_ATE=2
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Validação Externa/D12V - Validacao Pluviometrica Independente Caete/Figuras/Serie_Temporal_T_Tracuateua_D12V.png
/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Validação Externa/D12V - Validacao Pluviometrica Independente Caete/Figuras/Dispersao_T_Tracuateua_D12V.png
LOCAL_SEARCH_Bloco 10=0
LOCAL_SEARCH_Bloco 22=0
B10 HTTP 200 CONTENT_TYPE text/html; charset=utf-8 BYTES 919867 SHA256 2b2ebbdac5ee593c401579393427efc14d839c0ddf26fe48d648c783d25854ac
B10 HAS_SEMENTE False HAS_AMOSTRA False PREFIX '<!doctype html><html lang="en-US" dir="ltr"><head><base href="https://accounts.g'
B22 HTTP 200 CONTENT_TYPE text/html; charset=utf-8 BYTES 920364 SHA256 da4f033fba8c699006e3f516157d4c5da298bc0ad4f0c5aae244f2a553adf622
B22 HAS_SEMENTE False HAS_AMOSTRA False PREFIX '<!doctype html><html lang="e

In [22]:
import joblib
import gzip
import csv
import json
from pathlib import Path

b10_candidates = sorted(ROOT.rglob("Checkpoint_SHAP_Teste_Final.joblib"))
b10_samples = sorted(ROOT.rglob("Amostra_SHAP_Teste_Final.csv"))
b10_configs = sorted(ROOT.rglob("Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json"))
b22_configs = sorted(ROOT.rglob("Configuracao_Bloco22_V2_Interpretacao.json"))
print("B10_CHECKPOINTS", [str(p) for p in b10_candidates])
print("B10_SAMPLES", [str(p) for p in b10_samples])
print("B10_CONFIGS", [str(p) for p in b10_configs])
print("B22_CONFIGS", [str(p) for p in b22_configs])

for p in b10_configs + b22_configs:
    text = p.read_text(encoding="utf-8", errors="replace")
    print("CONFIG", p.name, "SHA256", sha256_file(p), "SEMENTE_LINES", [line for line in text.splitlines() if "sement" in line.lower() or "pandas" in line.lower()][:20])

for p in b10_candidates:
    obj = joblib.load(p)
    print("B10_CHECKPOINT", p.name, "TYPE", type(obj).__name__, "KEYS", sorted(obj) if isinstance(obj, dict) else None)
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (str, int, float, bool, type(None))):
                print("B10_VALUE", key, repr(value))
            elif hasattr(value, "shape"):
                print("B10_SHAPE", key, getattr(value, "shape", None), "COLUMNS", list(value.columns)[:20] if hasattr(value, "columns") else None)
            elif isinstance(value, (list, tuple, set, dict)):
                print("B10_CONTAINER", key, type(value).__name__, "LEN", len(value), "PREVIEW", repr(list(value)[:5]) if not isinstance(value, dict) else repr(dict(list(value.items())[:5])))
            else:
                print("B10_TYPE", key, type(value).__name__)

for p in b10_samples:
    with p.open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f, delimiter=";"))
    print("B10_SAMPLE", p.name, "SHA256", sha256_file(p), "ROWS", len(rows), "COLUMNS", list(rows[0]) if rows else [], "ROW_ID_HEAD", [r.get("row_id") for r in rows[:10]])

for p in sorted((ROOT / "Artefatos Operacionais" / "Fase B - Interpretacao dos Modelos").rglob("*Metadados*B22*.json")):
    print("B22_METADATA", p, p.read_text(encoding="utf-8", errors="replace"))

B10_CHECKPOINTS ['/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Checkpoints/Checkpoint_SHAP_Teste_Final.joblib']
B10_SAMPLES ['/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 10 V2 - SHAP Permutacao/Tabelas/Amostra_SHAP_Teste_Final.csv']
B10_CONFIGS ['/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json']
B22_CONFIGS ['/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Auditorias/Bloco 22/Configuracao_Bloco22_V2_Interpretacao.json', '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco22_V2_Interpretacao.json']
CONFIG Configuracao_Bloco10_V2_SHAP_Permutacao_Caete.json SHA256 ec356d79bada6cbc1bf3d500bbce125e91414270923b7e75ff654cad69a127c0 SEMENTE_LINES []
CONFIG Configuracao_Bloco22_V2_Interpretacao.json SHA256 387ed63ce03cc02f4f39ba9fe63c5c0df47cf3b0ffc4c4c317